In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils import decorate, underride, configure_plot_style, AIBM_COLORS, code_to_who_country, code_to_wef_country, write_html_table

configure_plot_style()

In [ ]:
def load_and_inventory(filename):
    """
    Load a WHO health indicator CSV file and print inventory information.
    
    Filters the data to include only records from year 2000 to 2019 (excludes 2020+)
    to avoid COVID-19 pandemic distortions, and country-level data (excludes regional aggregates).
    Adds a 'Country' column mapping country codes to country names using code_to_who_country.
    
    Parameters
    ----------
    filename : str
        Path to the CSV file containing WHO health indicator data.
        
    Returns
    -------
    df : pandas.DataFrame
        Filtered DataFrame containing country-level data from 2000-2019,
        with an additional 'Country' column.
    years : numpy.ndarray
        Array of unique years present in the filtered dataset.
    """
    # Filter to 2000-2019 to exclude COVID-19 pandemic years (2020+)
    df = pd.read_csv(filename).query('Year >= 2000 and Year <= 2019 and CountryCode == "COUNTRY"')
    
    # Add country name column using code_to_who_country mapping
    # The 'Country' column contains country codes (e.g., 'USA', 'GBR')
    # Rename 'Country' (codes) to 'Code', then add 'Country' (names)
    df = df.rename(columns={'Country': 'Code'})
    df['Country'] = df['Code'].map(code_to_who_country)
    
    print(df.shape)

    try:
        print(df['IndicatorCode'].unique())
        print(df['IndicatorName'].unique())
    except KeyError:
        pass

    sexes = df['Sex'].unique()
    print(sexes)
    print(df['Comments'].unique())
    years = df['Year'].unique()
    print(years)
    print(df['Country'].unique())
    
    return df, years

In [ ]:
from functools import reduce

def compute_gender_gap(df, value_col, sexes):
    """
    Return a DataFrame with separate columns for each sex, a gap column, and a midpoint column,
    handling cases where one or more sexes are missing.

    Parameters
    ----------
    df : pandas.DataFrame
        Must include 'Code' (country codes), 'Year', 'Sex', and the specified value_col.
        May also include 'Country' (country names) which will be preserved.
    value_col : str
        Name of the column containing the numeric value to compare between sexes.
    sexes : list of str
        List of values in the 'Sex' column, e.g. ['SEX_MLE', 'SEX_FMLE'].
        The first two entries are used to compute the gap (first - second) and midpoint (average).

    Returns
    -------
    df_by_sex : pandas.DataFrame
        Contains columns for each available sex, a gap column (Gap_{value_col} = first - second, i.e., Male - Female),
        and a midpoint column (Mid_{value_col} = average of first and second) if both sexes
        are present. Also includes 'Country' (added via mapping) and 'Year'.
    """
    # Determine which columns to keep
    # Base columns: Code (country codes), Year, and the value column (which will be renamed)
    # Country (country names) will be added later via mapping
    base_cols = ['Code', 'Year']
    
    dfs = []

    # Build a renamed DataFrame for each sex, only if it exists
    # Select only the columns we want to keep before merging
    for sex in sexes:
        temp = df[df['Sex'] == sex]
        if not temp.empty:
            # Select only the columns we need: base columns + value column
            cols_to_select = base_cols + [value_col]
            temp = temp[cols_to_select].copy()
            # Rename the value column to include the sex
            temp = temp.rename(columns={value_col: f"{value_col}_{sex}"})
            dfs.append(temp)

    # If no data at all, return empty DataFrame
    if not dfs:
        return pd.DataFrame(columns=['Code', 'Year'])

    # Merge all available sexes
    merge_on = ['Code', 'Year']
    df_by_sex = reduce(lambda left, right: left.merge(right, on=merge_on, how='outer'), dfs)

    # Compute the gap and midpoint only if both sexes are available
    # Gap = Male - Female (first - second, where first is typically Male)
    if len(sexes) >= 2:
        col1 = f"{value_col}_{sexes[0]}"
        col2 = f"{value_col}_{sexes[1]}"
        if col1 in df_by_sex.columns and col2 in df_by_sex.columns:
            df_by_sex[f"Gap_{value_col}"] = df_by_sex[col1] - df_by_sex[col2]
            df_by_sex[f"Mid_{value_col}"] = (df_by_sex[col1] + df_by_sex[col2]) / 2

    df_by_sex['Country'] = df_by_sex['Code'].map(code_to_who_country)

    return df_by_sex


In [ ]:
def summarize_gap(df, col, sexes=None):
    """
    Compute gender gap and create summary visualization using most recent data per country.
    
    Computes gender gaps using compute_gender_gap, then selects the most recent year
    available for each country (which may differ by country). Creates a scatter plot
    comparing values between sexes (if multiple sexes are provided) for OECD countries only.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing health indicator data with 'Code' (country codes), 'Year', 'Sex',
        and the value column specified by col. May also include 'Country' (country names).
    col : str
        Name of the column containing the numeric value to compare between sexes.
    sexes : list of str
        List of sex values to compare (e.g., ['Male', 'Female']).
        
    Returns
    -------
    df_gap : pandas.DataFrame
        DataFrame with gender gap computed for all years. Includes 'Country' and 'Year' columns.
    df_recent : pandas.DataFrame
        DataFrame with the most recent available year for each country, indexed by Code.
        Includes 'Country' and 'Year' columns. Returns all countries, not just OECD.
    """
    sexes = sexes or ['Male', 'Female']
    df_gap = compute_gender_gap(df, col, sexes)
    
    # Get the most recent year for each country
    most_recent_years = df_gap.groupby('Code')['Year'].max().reset_index()
    df_recent = df_gap.merge(most_recent_years, on=['Code', 'Year'])
    
    # Set index to Code, but keep Country and Year as columns
    df_recent = df_recent.set_index('Code')
    
    if len(sexes) > 1:
        cols = [f'{col}_{sex}' for sex in sexes]
        # Get OECD countries for plotting
        df_recent_oecd = get_oecd(df_recent)
        high = df_recent_oecd[cols].max().max()
        domain = [0, high]
        scatter_plot(df_recent_oecd, cols, domain)
        decorate(xlabel=f'{col}, Male', 
                 ylabel=f'{col}, Female', title=f'OECD Countries')
    
    return df_gap, df_recent

In [ ]:
def scatter_plot(df, cols, domain, **options):
    """
    Create a scatter plot comparing two columns with a diagonal reference line.
    
    Plots the values from two columns against each other, with a diagonal
    reference line (y=x) to show equality. Uses a square aspect ratio by default.
    Labels specific countries: USA, DEU, LTU, NDL.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame indexed by country, containing the columns to plot.
    cols : list of str
        List of two column names to plot on x and y axes.
    domain : list of float
        Two-element list [min, max] defining the plot domain for both axes.
    **options : dict
        Additional keyword arguments passed to decorate() for plot customization
        (e.g., xlabel, ylabel, title).
    """
    plt.plot(df[cols[0]], df[cols[1]], '.', color=AIBM_COLORS['crimson'])
    plt.plot(domain, domain, color='gray', alpha=0.5)
    
    # Label specific countries
    countries_to_label = ['USA', 'DEU', 'LTU', 'NLD']
    for country in countries_to_label:
        if country in df.index:
            x_val = df.loc[country, cols[0]]
            y_val = df.loc[country, cols[1]]
            if pd.notna(x_val) and pd.notna(y_val):
                plt.annotate(country, (x_val, y_val), 
                           xytext=(5, 5), textcoords='offset points',
                           fontsize=9, alpha=0.8)
    
    underride(options, aspect='equal')
    decorate(**options)

In [ ]:
from utils import oecd_codes
import warnings

def get_oecd(df):
    """
    Filter DataFrame to include only OECD member countries.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame indexed by country codes.
        
    Returns
    -------
    pandas.DataFrame
        Subset of the input DataFrame containing only OECD countries.
        
    Note
    -----
    Warns if any expected OECD countries are missing from the data.
    """
    available_countries = set(df.index)
    expected_oecd = set(oecd_codes)
    missing_countries = expected_oecd - available_countries
    
    if missing_countries:
        warnings.warn(
            f"Missing {len(missing_countries)} OECD countries in data: {sorted(missing_countries)}",
            UserWarning
        )
    
    return df.loc[df.index.intersection(oecd_codes)]

In [ ]:
def summarize_years(predictor_dfs):
    """
    Create a summary table of year coverage for each indicator in predictor_dfs.
    
    Parameters
    ----------
    predictor_dfs : dict
        Dictionary mapping indicator names to DataFrames containing predictor data.
        Each DataFrame should have a 'Year' column.
        
    Returns
    -------
    pandas.DataFrame
        Summary table with columns: Indicator, Low_Year, High_Year, Most_Common_Year,
        N_Countries_Most_Common, Total_Countries.
    """
    year_summary = []
    
    for name, df in predictor_dfs.items():
        years = df['Year'].dropna()
        if len(years) > 0:
            year_counts = years.value_counts()
            most_common_year = year_counts.index[0]
            n_countries_most_common = year_counts.iloc[0]
            
            year_summary.append({
                'Indicator': name,
                'Low_Year': int(years.min()),
                'High_Year': int(years.max()),
                'Most_Common_Year': int(most_common_year),
                'N_Countries_Most_Common': int(n_countries_most_common),
                'Total_Countries': len(df)
            })
        else:
            year_summary.append({
                'Indicator': name,
                'Low_Year': np.nan,
                'High_Year': np.nan,
                'Most_Common_Year': np.nan,
                'N_Countries_Most_Common': 0,
                'Total_Countries': len(df)
            })
            
    return pd.DataFrame(year_summary)

In [ ]:
from empiricaldist import Cdf

def plot_cdfs(df, label='', **options):
    """
    Plot cumulative distribution functions (CDFs) for columns ending in 'ale'.
    
    Creates CDF plots for all columns in the DataFrame that end with 'ale'
    (typically 'Male' and 'Female' columns). Each CDF is plotted with a label
    combining the column name and the provided label.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing numeric columns to plot as CDFs.
    label : str, optional
        Additional label text to append to each CDF plot label (default: '').
    **options : dict
        Additional keyword arguments for plot customization.
    """
    cols = [col for col in df.columns if col.endswith('ale')]
    for col in cols:
        vals = df[col].dropna()
        if vals.count() == 0:
            break
        cdf = Cdf.from_seq(df[col])
        cdf.plot(label=f'{col} {label}', **options)

In [ ]:
def plot_distributions(recent, **options):
    """
    Plot CDFs comparing all countries vs OECD countries.
    
    Creates cumulative distribution function plots for both all countries
    and OECD countries subset, allowing comparison of distributions.
    
    Parameters
    ----------
    recent : pandas.DataFrame
        DataFrame indexed by country codes, containing columns ending in 'ale'
        (typically 'Male' and 'Female' columns).
    **options : dict
        Additional keyword arguments passed to decorate() for plot customization
        (e.g., xlabel, title). The ylabel is automatically set to 'CDF'.
    """
    plot_cdfs(recent, label='All countries')
    plot_cdfs(get_oecd(recent), label='OECD')
    
    # Extract indicator name from column names (remove _Male or _Female suffix)
    cols = [col for col in recent.columns if col.endswith('ale')]
    if cols:
        # Get the first column and extract the indicator name
        first_col = cols[0]
        # Remove _Male or _Female suffix to get the indicator name
        indicator_name = first_col.rsplit('_')[:-1] if '_' in first_col else first_col
        xlabel = indicator_name
    else:
        xlabel = ''
    
    underride(options, xlabel=xlabel, ylabel='CDF')
    decorate(**options)

In [ ]:
# Map from original value column names to short indicator names
# Used to rename columns before computing gaps so Gap_ and Mid_ columns use short names
column_name_mapping = {
    'CardioDeathRate': 'Cardiovascular',
    'ChronicRespiratoryDeathRate': 'ChronicRespiratory',
    'SuicideRate': 'Suicide',
    'AlcoholDeathRate': 'Alcohol',
    'PoisoningRate': 'Poisoning',
    'RoadTrafficDeathRate': 'RoadTraffic',
    'HomicideRate': 'Homicide',
    'MaternalMortalityRatio': 'MaternalMortality',
    'U5MR': 'Childhood',
    'DiabetesDeathRate': 'Diabetes',
    'DrugDisorderDeathRate': 'DrugDisorder',
    'UnintentionalInjuriesDeathRate': 'UnintentionalInjury',
    'NeoplasmsDeathRate': 'Neoplasms',
}

## WHO HALE data

**Healthy Life Expectancy (HALE) at birth** - The average number of years that a person can expect to live in "full health" by taking into account years lived in less than full health due to disease and/or injury. This is the **target variable** for the analysis. The gender gap (Male HALE - Female HALE) measures the difference in healthy life expectancy between men and women.

**Indicator Code**: WHOSIS_000002  
**Relevance**: Direct measure of the outcome we're trying to explain. Gender differences in HALE reflect the cumulative impact of all mortality and morbidity factors that differentially affect men and women.

Downloaded using the GHO OData API (who_data.py)

https://www.who.int/data/gho/info/gho-odata-api

In [ ]:
filename = '../data/who_hale_data.csv'
hale, years = load_and_inventory(filename)

In [ ]:
d = {'SEX_BTSX': 'Both', 'SEX_FMLE': 'Female', 'SEX_MLE': 'Male', }
hale['Sex'] = hale['Sex'].replace(d)

In [ ]:
hale.head()

In [ ]:
col = 'HALE_Years'
year = years[-1]
hale_gap, hale_recent = summarize_gap(hale, col)

In [ ]:
plot_distributions(hale_recent)

## WHO Life Expectancy data

**Life Expectancy at birth** - The average number of years that a person can expect to live, regardless of health status. This is the **secondary target variable** for the analysis, allowing comparison of which factors explain the gender gap in overall life expectancy versus healthy life expectancy. The gender gap (Female LE - Male LE) measures the difference in life expectancy between women and men.

**Indicator Code**: WHOSIS_000001  
**Relevance**: Life expectancy captures all years lived (healthy and unhealthy), while HALE focuses on healthy years only. Both are calculated from birth, so both should be affected by the same mortality patterns. The relative importance of early-life vs adult mortality may differ between the two outcomes.

Downloaded using the GHO OData API (who_data.py)

https://www.who.int/data/gho/info/gho-odata-api

In [ ]:
filename = '../data/who_life_expectancy_data.csv'
le, years = load_and_inventory(filename)

In [ ]:
d = {'SEX_BTSX': 'Both', 'SEX_FMLE': 'Female', 'SEX_MLE': 'Male', }
le['Sex'] = le['Sex'].replace(d)

In [ ]:
le.head()

In [ ]:
col = 'LifeExpectancy_Years'
year = years[-1]
le_gap, le_recent = summarize_gap(le, col)

In [ ]:
plot_distributions(le_recent)

## Smoking

**Age-standardized current tobacco smoking prevalence (%)** - Percentage of population aged 15+ who currently smoke any tobacco product, age-standardized for cross-country comparison.

**Indicator Code**: M_Est_smk_curr_std  
**Relevance**: Historically, men have had significantly higher smoking rates than women. Smoking is a major contributor to cardiovascular disease, lung cancer, and respiratory diseases. As smoking rates have converged between genders in some countries, the life expectancy gap has narrowed, suggesting smoking is one of the most important modifiable factors contributing to the HALE gender gap.

In [ ]:
filename = '../data/who_smoking_data.csv'
smoking, years = load_and_inventory(filename)

In [ ]:
smoking.head()

In [ ]:
col = 'SmokingPrevalence'
year = years[-1]
smoking_gap, smoking_recent = summarize_gap(smoking, col)

In [ ]:
plot_distributions(smoking_recent)

## Suicide

**Age-standardized suicide rates (per 100,000 population)** - Deaths from intentional self-harm, age-standardized for cross-country comparison.

**Indicator Code**: MH_12  
**Relevance**: Suicide rates are typically higher in men across most countries, directly contributing to the gender gap in mortality. Suicide reflects mental health and social factors that differentially affect men and women, and is strongly linked to mortality.

In [ ]:
filename = '../data/who_suicide_rates.csv'
suicide, years = load_and_inventory(filename)

In [ ]:
suicide.head()

In [ ]:
col = 'SuicideRate'
suicide = suicide.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
suicide_gap, suicide_recent = summarize_gap(suicide, col)

In [ ]:
plot_distributions(suicide_recent)

## Alcohol

**Alcohol-attributable all-cause deaths per 100,000 (age-standardized)** - Deaths from all causes that are attributable to alcohol consumption, including direct alcohol-related deaths and alcohol-attributable deaths from other causes (e.g., accidents, liver disease).

**Indicator Code**: SA_0000001832  
**Relevance**: Men typically have higher rates of alcohol consumption and alcohol-related diseases. Alcohol contributes to liver disease, accidents, and various health conditions, directly impacting mortality. Age-standardized rates match HALE methodology for cross-country comparison.

In [ ]:
filename = '../data/who_alcohol_death_rates.csv'
alcohol, years = load_and_inventory(filename)

In [ ]:
alcohol.head()

In [ ]:
col = 'AlcoholDeathRate'
alcohol = alcohol.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
alcohol_gap, alcohol_recent = summarize_gap(alcohol, col)

In [ ]:
plot_distributions(alcohol_recent)

## Poison

**Mortality rate attributed to unintentional poisoning (per 100,000 population)** - Deaths from accidental poisonings from chemicals, drugs, and other substances.

**Indicator Code**: SDGPOISON  
**Relevance**: Men often have higher rates of accidental deaths, including poisonings. This reflects occupational hazards and risk-taking behaviors that contribute to the gender gap in mortality. Has excellent temporal coverage (2000-2021) and country coverage (196 countries).

In [ ]:
filename = '../data/who_poisoning_rates.csv'
poison, years = load_and_inventory(filename)

In [ ]:
poison.head()

In [ ]:
col = 'PoisoningRate'
poison = poison.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
poison_gap, poison_recent = summarize_gap(poison, col)

In [ ]:
plot_distributions(poison_recent)

## Traffic

**Road traffic crash deaths, age-standardized death rates (15+), per 100,000 population** - Deaths from road traffic accidents, age-standardized for ages 15+.

**Indicator Code**: SA_0000001459  
**Relevance**: Road traffic deaths are typically 2-4 times higher in men across most countries, making it a major contributor to the gender gap in mortality. Reflects higher exposure to driving (including occupational exposure), occupational hazards, and potentially risk-taking behaviors. Age-standardized rates for ages 15+ match HALE methodology.

In [ ]:
filename = '../data/who_road_traffic_death_rates.csv'
traffic, years = load_and_inventory(filename)

In [ ]:
traffic.head()

In [ ]:
col = 'RoadTrafficDeathRate'
traffic = traffic.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
traffic_gap, traffic_recent = summarize_gap(traffic, col)

In [ ]:
plot_distributions(traffic_recent)

## Maternal mortality

**Maternal mortality ratio (per 100,000 live births)** - Deaths of women during pregnancy, childbirth, or within 42 days of termination of pregnancy, per 100,000 live births.

**Indicator Code**: MDG_0000000026  
**Relevance**: Critical for understanding cases where the HALE gender gap is small due to high female mortality, especially in lower-income countries. High maternal mortality can significantly reduce the HALE gender gap by lowering female life expectancy. Inherently female-specific, so only female values are used in analysis.

In [ ]:
filename = '../data/who_maternal_mortality_ratio.csv'
maternal, years = load_and_inventory(filename)

In [ ]:
maternal.head()

In [ ]:
col = 'MaternalMortalityRatio'
maternal = maternal.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
maternal_gap, maternal_recent = summarize_gap(maternal, col, sexes=['Female'])

In [ ]:
plot_distributions(maternal_recent)

## Homicide

**Estimates of rates of homicides per 100,000 population** - Deaths from intentional homicide, including estimates with confidence intervals.

**Indicator Code**: VIOLENCE_HOMICIDERATE  
**Relevance**: Homicide rates are typically much higher in men across most countries, making it a major contributor to the gender gap in mortality. Homicide reflects violence, conflict, and social factors that differentially affect men and women. Has excellent temporal coverage (2000-2021) and country coverage (196 countries).

In [ ]:
filename = '../data/who_homicide_rates.csv'
homicide, years = load_and_inventory(filename)

In [ ]:
homicide.head()

In [ ]:
col = 'HomicideRate'
homicide = homicide.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
homicide_gap, homicide_recent = summarize_gap(homicide, col)

In [ ]:
plot_distributions(homicide_recent)

## Intimate Partner Violence

**Proportion of ever-partnered women and girls aged 15-49 years subjected to physical and/or sexual violence by a current or former intimate partner in the previous 12 months (%)** - Prevalence indicator measuring the percentage of women experiencing intimate partner violence.

**Indicator Code**: SDGIPV  
**Relevance**: Note: This is a **prevalence indicator** (percentage), not a direct death rate. IPV affects women's health indirectly through mental health impacts, injuries, and other health consequences. It may contribute to the gender gap in HALE through its effects on women's physical and mental health, though the relationship is complex and indirect. Inherently female-specific, so only female values are used in analysis.

In [ ]:
filename = '../data/who_ipv_prevalence.csv'
ipv, years = load_and_inventory(filename)

In [ ]:
ipv.head()

In [ ]:
col = 'IPVPrevalence'
year = years[-1]
ipv_gap, ipv_recent = summarize_gap(ipv, col, sexes=['Female'])

In [ ]:
plot_distributions(ipv_recent)

## Under five mortality rate

**Under-five mortality rate (probability of dying by age 5 per 1000 live births)** - Deaths of children under age 5 per 1,000 live births, with gender breakdowns.

**Indicator Code**: MDG_0000000007  
**Relevance**: HALE is calculated from birth, so under-five mortality directly affects HALE calculations. If child mortality differs by gender, it directly contributes to the HALE gender gap. Infant mortality is typically higher in males (biological vulnerability + some behavioral factors). More important in lower-income countries with high child mortality. Note: MDG_0000000007 chosen over u5mr for better data quality when filtered for sex dimension.

In [ ]:
filename = '../data/who_u5mr.csv'
u5mr, years = load_and_inventory(filename)

In [ ]:
u5mr.head()

In [ ]:
col = 'U5MR'
u5mr = u5mr.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
u5mr_gap, u5mr_recent = summarize_gap(u5mr, col)

In [ ]:
plot_distributions(u5mr_recent)

## Cardiovascular Disease

**Age-standardized cardiovascular disease death rates (per 100,000)** - Deaths from cardiovascular diseases (heart disease, stroke, etc.), age-standardized for cross-country comparison.

**Indicator Code**: Multiple codes tried (WHS2_161, etc.) - see `who_data.py` for implementation details  
**Relevance**: Men typically have higher rates of cardiovascular disease and heart attacks, contributing significantly to the gender gap in mortality. Risk factors include smoking, diet, and potentially biological differences. May capture effects of smoking and other risk factors. Age-standardized rates match HALE methodology.

In [ ]:
filename = '../data/who_cardiovascular_death_rates.csv'
cardio, years = load_and_inventory(filename)

In [ ]:
# Rename DeathRate to CardioDeathRate to avoid ambiguity
cardio = cardio.rename(columns={'DeathRate': 'CardioDeathRate'})
cardio.head()

In [ ]:
col = 'CardioDeathRate'
year = years[-1]
cardio_gap, cardio_recent = summarize_gap(cardio, col)

In [ ]:
plot_distributions(cardio_recent)

## Diabetes

**Age-standardized death rates, diabetes mellitus (per 100,000)** - Deaths from diabetes, age-standardized for cross-country comparison.

**Indicator Code**: SA_0000001440  
**Relevance**: Diabetes is a chronic condition that can contribute to the gender gap in mortality, though the relationship may vary by country and healthcare access. Age-standardized rates match HALE methodology. **Limitation**: Only has data for 2004 (similar to cardiovascular disease indicators), which limits temporal analysis but provides a good cross-sectional snapshot.

In [ ]:
filename = '../data/who_diabetes_death_rates.csv'
diabetes, years = load_and_inventory(filename)

In [ ]:
diabetes.head()

In [ ]:
col = 'DiabetesDeathRate'
year = years[-1]
diabetes_gap, diabetes_recent = summarize_gap(diabetes, col)

In [ ]:
plot_distributions(diabetes_recent)

## NCD Mortality (30-70 years)

**Probability (%) of dying between age 30 and exact age 70 from any of cardiovascular disease, cancer, diabetes, or chronic respiratory disease** - Combined non-communicable disease mortality indicator.

**Indicator Code**: NCDMORT3070  
**Relevance**: Combines multiple causes of death (cardiovascular disease, cancer, diabetes, chronic respiratory disease), so it's less specific than individual cause indicators. However, it has much better temporal coverage (2000-2021) than diabetes-specific indicators (which only have 2004 data). This makes it useful for model comparison - trading off specificity for temporal coverage. The combined indicator may capture overall NCD mortality patterns that contribute to the HALE gender gap.

In [ ]:
filename = '../data/who_ncd_mortality_30_70.csv'
ncdmort, years = load_and_inventory(filename)

In [ ]:
ncdmort.head()

In [ ]:
col = 'NCDMortality30_70'
year = years[-1]
ncdmort_gap, ncdmort_recent = summarize_gap(ncdmort, col)

In [ ]:
plot_distributions(ncdmort_recent)

## Drug Use Disorders (IHME)

**Drug use disorder death rates (per 100,000 population)** - Deaths from drug use disorders, including overdoses, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Drug overdoses, particularly opioid overdoses, are a major cause of death in some OECD countries (especially the US) and may contribute significantly to the HALE gender gap. This indicator captures overdose deaths that may not be fully captured in the WHO poisoning indicator. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
def load_ihme_indicator(filename_male, filename_female, value_col_name, indicator_code, indicator_name):
    """
    Load IHME indicator data from separate male and female files and convert to WHO-compatible format.
    
    Converts IHME CSV format (Location=country name, Sex="Male"/"Female") to WHO format
    (Country=country code, Sex="Male"/"Female", etc.). Filters to 2000-2019.
    
    Parameters
    ----------
    filename_male : str
        Path to the IHME CSV file with male data.
    filename_female : str
        Path to the IHME CSV file with female data.
    value_col_name : str
        Name for the value column (e.g., 'DrugDisorderDeathRate', 'DiabetesDeathRate').
    indicator_code : str
        Indicator code for the IHME indicator (e.g., 'IHME_DRUG_DISORDERS', 'IHME_DIABETES_TYPE2').
    indicator_name : str
        Human-readable indicator name (e.g., 'Drug use disorders, death rate per 100,000').
        
    Returns
    -------
    df : pandas.DataFrame
        DataFrame in WHO-compatible format with columns: IndicatorCode, IndicatorName,
        Code, CountryCode, Year, Sex, value_col_name, value_col_name_Low, 
        value_col_name_High, Country.
    years : numpy.ndarray
        Array of unique years present in the filtered dataset.
    """
    # Create reverse mapping from country name to code
    who_country_to_code = {country: code for code, country in code_to_who_country.items()}
    
    # Map IHME country names that differ from WHO names
    ihme_country_name_mapping = {
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States'
    }
    
    def process_ihme_file(filename, sex_value):
        """Helper function to process a single IHME file."""
        df = pd.read_csv(filename)
        
        # Filter to 2000-2019 (exclude 2020+ for COVID-19 reasons)
        df = df.query('Year >= 2000 and Year <= 2019')
        
        # Filter to "All ages" (if Age column exists)
        if 'Age' in df.columns:
            df = df.query('Age == "All ages"')
        
        # Map IHME country names to WHO country names
        df['Location'] = df['Location'].replace(ihme_country_name_mapping)
        
        # Convert country names to codes
        df['Code'] = df['Location'].map(who_country_to_code)
        
        # Filter out rows where country mapping failed (not in our country list)
        df = df[df['Code'].notna()].copy()
        
        # Set Sex column to the specified value (Male or Female)
        df['Sex'] = sex_value
        
        # Rename and create columns to match WHO format
        df['IndicatorCode'] = indicator_code
        df['IndicatorName'] = indicator_name
        df['CountryCode'] = 'COUNTRY'
        df[value_col_name] = df['Value']
        df[f'{value_col_name}_Low'] = df['Lower bound']
        df[f'{value_col_name}_High'] = df['Upper bound']
        df['Country'] = df['Location']
        
        # Select and reorder columns to match WHO format
        columns_to_keep = [
            'IndicatorCode', 'IndicatorName', 'Code', 'CountryCode', 'Year', 'Sex',
            value_col_name, f'{value_col_name}_Low', f'{value_col_name}_High',
            'Country'
        ]
        df = df[columns_to_keep].copy()
        
        return df
    
    # Load and process both files
    df_male = process_ihme_file(filename_male, 'Male')
    df_female = process_ihme_file(filename_female, 'Female')
    
    # Concatenate male and female data
    df = pd.concat([df_male, df_female], ignore_index=True)
    
    # Sort by country, sex, and year
    df = df.sort_values(['Country', 'Sex', 'Year']).reset_index(drop=True)
    
    years = df['Year'].unique()
    
    print(df.shape)
    print(f"Years: {years.min():.0f} - {years.max():.0f}")
    print(f"Countries: {df['Country'].nunique()}")
    print(f"Sex categories: {df['Sex'].unique()}")
    
    return df, years

In [ ]:
filename_male = '../data/ihme_drug_disorder_deaths_male.csv'
filename_female = '../data/ihme_drug_disorder_deaths_female.csv'
drug_disorders, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='DrugDisorderDeathRate',
    indicator_code='IHME_DRUG_DISORDERS',
    indicator_name='Drug use disorders, death rate per 100,000'
)

In [ ]:
drug_disorders.head()

In [ ]:
col = 'DrugDisorderDeathRate'
drug_disorders = drug_disorders.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
drug_disorders_gap, drug_disorders_recent = summarize_gap(drug_disorders, col)

In [ ]:
plot_distributions(drug_disorders_recent)

## Diabetes Type 2 (IHME)

**Diabetes type 2 death rates (per 100,000 population)** - Deaths from diabetes mellitus type 2, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: This is an alternative to the WHO diabetes death rate indicator (SA_0000001440) which only has data for 2004. IHME data may have better temporal coverage, allowing for more recent data to be used in the analysis. Diabetes is a chronic condition that can contribute to the gender gap in mortality, though the relationship may vary by country and healthcare access. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_diabetes_deaths_male.csv'
filename_female = '../data/ihme_diabetes_deaths_female.csv'
diabetes_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='DiabetesDeathRate',
    indicator_code='IHME_DIABETES_TYPE2',
    indicator_name='Diabetes mellitus type 2, death rate per 100,000'
)

In [ ]:
diabetes_ihme.head()

In [ ]:
col = 'DiabetesDeathRate'
diabetes_ihme = diabetes_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
diabetes_ihme_gap, diabetes_ihme_recent = summarize_gap(diabetes_ihme, col)

In [ ]:
plot_distributions(diabetes_ihme_recent)

## Cardiovascular Diseases (IHME)

**Cardiovascular diseases death rates (per 100,000 population)** - Deaths from cardiovascular diseases, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Cardiovascular diseases are a major cause of death and may contribute significantly to the HALE gender gap. This is an alternative to the WHO cardiovascular disease death rate indicators which only have data for 2004. IHME data may have better temporal coverage, allowing for more recent data to be used in the analysis. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_cardiovascular_deaths_male.csv'
filename_female = '../data/ihme_cardiovascular_deaths_female.csv'
cardiovascular_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='CardioDeathRate',
    indicator_code='IHME_CARDIOVASCULAR',
    indicator_name='Cardiovascular diseases, death rate per 100,000'
)

In [ ]:
cardiovascular_ihme.head()

In [ ]:
col = 'CardioDeathRate'
cardiovascular_ihme = cardiovascular_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
cardiovascular_ihme_gap, cardiovascular_ihme_recent = summarize_gap(cardiovascular_ihme, col)

In [ ]:
plot_distributions(cardiovascular_ihme_recent)

## Neoplasms (Cancer) (IHME)

**Neoplasms (cancer) death rates (per 100,000 population)** - Deaths from neoplasms (cancer), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Neoplasms (cancer) are a major cause of death and may contribute significantly to the HALE gender gap. Different types of cancer have different gender patterns (e.g., lung cancer is often higher in men, breast cancer is female-specific). This indicator provides comprehensive cancer death rates with better temporal coverage than WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_neoplasms_deaths_male.csv'
filename_female = '../data/ihme_neoplasms_deaths_female.csv'
neoplasms_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='NeoplasmsDeathRate',
    indicator_code='IHME_NEOPLASMS',
    indicator_name='Neoplasms (cancer), death rate per 100,000'
)

In [ ]:
neoplasms_ihme.head()

In [ ]:
col = 'NeoplasmsDeathRate'
neoplasms_ihme = neoplasms_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
neoplasms_ihme_gap, neoplasms_ihme_recent = summarize_gap(neoplasms_ihme, col)

In [ ]:
plot_distributions(neoplasms_ihme_recent)

## Chronic Respiratory Diseases (IHME)

**Chronic respiratory diseases death rates (per 100,000 population)** - Deaths from chronic respiratory diseases (including COPD, asthma, and other chronic lung conditions), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Chronic respiratory diseases are a major cause of death and may contribute significantly to the HALE gender gap. These diseases often have gender differences due to factors such as smoking patterns, occupational exposures, and environmental factors. This indicator provides comprehensive chronic respiratory disease death rates with better temporal coverage than WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_chronic_respiratory_deaths_male.csv'
filename_female = '../data/ihme_chronic_respiratory_deaths_female.csv'
chronic_respiratory_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='ChronicRespiratoryDeathRate',
    indicator_code='IHME_CHRONIC_RESPIRATORY',
    indicator_name='Chronic respiratory diseases, death rate per 100,000'
)

In [ ]:
chronic_respiratory_ihme.head()

In [ ]:
col = 'ChronicRespiratoryDeathRate'
chronic_respiratory_ihme = chronic_respiratory_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
chronic_respiratory_ihme_gap, chronic_respiratory_ihme_recent = summarize_gap(chronic_respiratory_ihme, col)

In [ ]:
plot_distributions(chronic_respiratory_ihme_recent)

## Unintentional Injuries (IHME)

**Unintentional injuries death rates (per 100,000 population)** - Deaths from unintentional injuries (including falls, drowning, fires, and other accidents), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Unintentional injuries are a significant cause of death and may contribute to the HALE gender gap. These injuries often show gender differences due to occupational exposures, risk-taking behaviors, and activity patterns. This indicator provides comprehensive unintentional injury death rates with better temporal coverage (1990-2023) than many WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_unintentional_injuries_deaths_male.csv'
filename_female = '../data/ihme_unintentional_injuries_deaths_female.csv'
unintentional_injuries_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='UnintentionalInjuriesDeathRate',
    indicator_code='IHME_UNINTENTIONAL_INJURIES',
    indicator_name='Unintentional injuries, death rate per 100,000'
)

In [ ]:
unintentional_injuries_ihme.head()

In [ ]:
col = 'UnintentionalInjuriesDeathRate'
unintentional_injuries_ihme = unintentional_injuries_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
year = years[-1]
unintentional_injuries_ihme_gap, unintentional_injuries_ihme_recent = summarize_gap(unintentional_injuries_ihme, col)

In [ ]:
plot_distributions(unintentional_injuries_ihme_recent)

## Phase 1: Data Preparation for Regression Analysis

### Step 1.2: Prepare Target Variables (HALE and Life Expectancy Gender Gaps)

In [ ]:
# Calculate HALE gender gap from existing hale_recent DataFrame
# Gap = Female - Male (positive means females have higher HALE)
hale_recent['HALE_gap'] = hale_recent['HALE_Years_Female'] - hale_recent['HALE_Years_Male']

# Filter to OECD countries
hale_oecd = get_oecd(hale_recent)

# Display summary
hale_oecd[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].describe()

In [ ]:
# Calculate Life Expectancy gender gap from existing le_recent DataFrame
# Gap = Female - Male (positive means females have higher Life Expectancy)
le_recent['LifeExpectancy_gap'] = le_recent['LifeExpectancy_Years_Female'] - le_recent['LifeExpectancy_Years_Male']

# Filter to OECD countries
le_oecd = get_oecd(le_recent)

# Display summary
le_oecd[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].describe()

In [ ]:
# Diagnostic: Check Israel's (ISR) HALE values
if 'ISR' in hale_recent.index:
    isr_data = hale_recent.loc[['ISR']]
    print("=== Israel (ISR) HALE Data ===")
    print(f"Country Name: {isr_data['Country'].iloc[0]}")
    
    # Check if this is an outlier
    print(f"\nHALE Gap: {isr_data['HALE_gap'].iloc[0]:.2f} years")
    print(f"Male HALE: {isr_data['HALE_Years_Male'].iloc[0]:.2f} years")
    print(f"Female HALE: {isr_data['HALE_Years_Female'].iloc[0]:.2f} years")
    
    # Compare to other countries
    print(f"\nCountries with negative HALE gap (men > women):")
    negative_gap = hale_recent[hale_recent['HALE_gap'] < 0]
    if len(negative_gap) > 0:
        negative_gap[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].sort_values('HALE_gap')
    else:
        print("No other countries with negative gap found")
    
    # Check raw HALE data for Israel to see if there's a data issue
    print(f"\n=== Checking raw HALE data for Israel ===")
    if 'ISR' in hale.index:
        isr_raw = hale[hale['Code'] == 'ISR'].sort_values('Year')
        print(f"Years available: {sorted(isr_raw['Year'].unique())}")
        print(f"\nRaw data by year and sex:")
        isr_raw[['Year', 'Sex', 'HALE_Years']].pivot(index='Year', columns='Sex', values='HALE_Years')
    else:
        isr_data[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']]
else:
    print("Israel (ISR) not found in hale_recent")

### Step 1.4: Merge All Predictors into Single Dataset

In [ ]:
# Start with HALE data as base (we'll add LE later)
analysis_df = hale_oecd[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].copy()

# Add Life Expectancy data
analysis_df = analysis_df.join(le_oecd[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']], how='outer')

# Merge all predictor DataFrames (already filtered to OECD in earlier sections)
# Note: We will exclude Male/Female columns, Country, and Year columns before merging
# We keep Mid (midpoint) and Gap columns as predictors instead of separate Male/Female columns
predictor_dfs = {
    'Alcohol': get_oecd(alcohol_recent),
    'ChronicRespiratory': get_oecd(chronic_respiratory_ihme_recent),
    'UnintentionalInjury': get_oecd(unintentional_injuries_ihme_recent),
    'RoadTraffic': get_oecd(traffic_recent),
    'Diabetes': get_oecd(diabetes_ihme_recent),
    'Cardiovascular': get_oecd(cardiovascular_ihme_recent),
    'Childhood': get_oecd(u5mr_recent),
    'DrugDisorder': get_oecd(drug_disorders_recent),
    'Homicide': get_oecd(homicide_recent),
    'Poisoning': get_oecd(poison_recent),
    'Suicide': get_oecd(suicide_recent),
    'MaternalMortality': get_oecd(maternal_recent),
    'Neoplasms': get_oecd(neoplasms_ihme_recent),
}

In [ ]:
# Summary table of year coverage for each indicator
year_summary_df = summarize_years(predictor_dfs)
year_summary_df

In [ ]:
for name, df in predictor_dfs.items():
    drop_cols = ['Country', 'Year']
    # Keep Male, Female, Mid, and Gap columns for counterfactual analysis
    # Only drop Country and Year columns
    if drop_cols:
        predictor_dfs[name] = df.drop(columns=drop_cols)

# Check shapes after dropping Country and Year
for name, predictor_df in predictor_dfs.items():
    print(name, predictor_df.shape)

In [ ]:
# Merge all predictors on index (Country codes)
for name, df in predictor_dfs.items():
    analysis_df = analysis_df.join(df, how='outer')

# Display shape and column names
analysis_df.shape

In [ ]:
analysis_df.head()

In [ ]:
# Create missing data report
missing_report = pd.DataFrame({
    'Indicator': analysis_df.columns,
    'Missing_Count': [analysis_df[col].isna().sum() for col in analysis_df.columns],
    'Missing_Pct': [analysis_df[col].isna().sum() / len(analysis_df) * 100 for col in analysis_df.columns],
    'Available_Count': [analysis_df[col].notna().sum() for col in analysis_df.columns]
}).sort_values('Missing_Count', ascending=False)

missing_report

In [ ]:
# Show which countries have complete data for all indicators
complete_cases = analysis_df.dropna()
complete_cases.shape[0], f"{complete_cases.shape[0] / len(analysis_df) * 100:.1f}% of countries have complete data"

In [ ]:
#bad = analysis_df['IPVPrevalence_Female'].isna()
#analysis_df.loc[bad].index.map(code_to_who_country)

### Step 1.5: Create Final Analysis Dataset

In [ ]:
# Use complete-case analysis for primary model
analysis_complete = analysis_df.dropna()

# Document excluded countries
excluded_countries = set(analysis_df.index) - set(analysis_complete.index)
excluded_countries if excluded_countries else "No countries excluded - all OECD countries have complete data"

In [ ]:
# Separate target and predictors
# Create both target variables
target_hale = analysis_complete['HALE_gap']
target_le = analysis_complete['LifeExpectancy_gap']
# Keep all predictor columns including Male, Female, Mid, and Gap for counterfactual analysis
# Only drop the target variable columns
predictors = analysis_complete.drop(columns=[
    'HALE_gap', 'HALE_Years_Male', 'HALE_Years_Female',
    'LifeExpectancy_gap', 'LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female'
])

# Display final dataset info
pd.DataFrame({
    'Dataset': ['Complete Cases'],
    'Countries': [len(analysis_complete)],
    'Target_Variables': ['HALE_gap, LifeExpectancy_gap'],
    'Number_of_Predictors': [len(predictors.columns)],
    'Predictor_Names': [', '.join(predictors.columns)]
})

In [ ]:
# Summary of Phase 1 completion
pd.DataFrame({
    'Step': ['1.2: Target Variable', '1.4: Merge', '1.5: Complete Cases'],
    'Status': ['Complete', 'Complete', 'Complete'],
    'Countries': [len(hale_oecd), len(analysis_df), len(analysis_complete)],
    'Variables': [3, len(analysis_df.columns), len(analysis_complete.columns)]
})

**Note**: Predictor standardization will be done as part of the regression pipeline (e.g., using `StandardScaler` in scikit-learn's pipeline), not as a separate preprocessing step.

## Phase 2: Exploratory Data Analysis

### Step 2.1: Descriptive Statistics

#### HALE Gender Gap (Primary Target Variable)

In [ ]:
target_hale.sort_values()

In [ ]:
# Summary statistics for target variable (HALE gap)
target_hale.describe()

In [ ]:
# Distribution of HALE gap across OECD countries
plt.hist(target_hale, bins=15, color=AIBM_COLORS['crimson'], edgecolor='white')
decorate(xlabel='HALE Gap (Female - Male, years)', 
         ylabel='Number of Countries',
         title='Distribution of HALE Gender Gap Across OECD Countries')

In [ ]:
# Identify potential outliers using IQR method
Q1 = target_hale.quantile(0.25)
Q3 = target_hale.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = target_hale[(target_hale < lower_bound) | (target_hale > upper_bound)]
outliers_df = pd.DataFrame({
    'Country': outliers.index,
    'HALE_Gap': outliers.values
}).sort_values('HALE_Gap')

outliers_df if not outliers_df.empty else "No outliers detected using IQR method"

#### Life Expectancy Gender Gap (Secondary Target Variable)

In [ ]:
target_le.sort_values()

In [ ]:
# Summary statistics for target variable (Life Expectancy gap)
target_le.describe()

In [ ]:
# Distribution of Life Expectancy gap across OECD countries
plt.hist(target_le, bins=15, color=AIBM_COLORS['blue'], edgecolor='white')
decorate(xlabel='Life Expectancy Gap (Female - Male, years)', 
         ylabel='Number of Countries',
         title='Distribution of Life Expectancy Gender Gap Across OECD Countries')

In [ ]:
# Identify potential outliers using IQR method
Q1 = target_le.quantile(0.25)
Q3 = target_le.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = target_le[(target_le < lower_bound) | (target_le > upper_bound)]
outliers_df = pd.DataFrame({
    'Country': outliers.index,
    'LifeExpectancy_Gap': outliers.values
}).sort_values('LifeExpectancy_Gap')

outliers_df if not outliers_df.empty else "No outliers detected using IQR method"

#### Comparison of HALE and Life Expectancy Gaps

In [ ]:
# Compare the two target variables
comparison_df = pd.DataFrame({
    'HALE_Gap': target_hale,
    'LifeExpectancy_Gap': target_le
})

comparison_df.describe()

In [ ]:
# Scatter plot comparing HALE gap vs Life Expectancy gap
plt.scatter(target_hale, target_le, color=AIBM_COLORS['crimson'], alpha=0.6)
decorate(xlabel='HALE Gap (Female - Male, years)', 
         ylabel='Life Expectancy Gap (Female - Male, years)',
         title='HALE Gap vs Life Expectancy Gap Across OECD Countries')

In [ ]:
# Correlation between the two target variables
correlation = target_hale.corr(target_le)
print(f"Correlation between HALE gap and Life Expectancy gap: {correlation:.3f}")

#### Summary Statistics for Predictors

In [ ]:
# Summary statistics for all predictors
predictors.describe()

### Step 2.2: Correlation Analysis

In [ ]:
# Calculate correlation matrix of all predictors
correlation_matrix = predictors.corr()

# Display correlation matrix
correlation_matrix

In [ ]:
# Visualize correlation matrix as heatmap
import seaborn as sns

plt.figure(figsize=(12, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))  # Mask upper triangle
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
decorate(title='Correlation Matrix of Predictors (Lower Triangle)')

In [ ]:
# Identify highly correlated predictor pairs (|correlation| > 0.7)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                'Predictor_1': correlation_matrix.columns[i],
                'Predictor_2': correlation_matrix.columns[j],
                'Correlation': corr_val
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
high_corr_df if not high_corr_df.empty else "No highly correlated pairs (|r| > 0.7) found"

In [ ]:
# Summary of correlation analysis
pd.DataFrame({
    'Analysis': ['Total Predictors', 'High Correlations (|r| > 0.7)', 'Max Correlation', 'Min Correlation'],
    'Value': [
        len(predictors.columns),
        len(high_corr_pairs) if high_corr_pairs else 0,
        correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].max(),
        correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].min()
    ]
})

### Step 2.3: Extreme Values and Country Rankings

#### For Each Indicator: Highest and Lowest Values

In [ ]:
# Map from short indicator names to their _recent dataframe variable names
indicator_to_recent_df = {
    'Cardiovascular': 'cardiovascular_ihme_recent',
    'ChronicRespiratory': 'chronic_respiratory_ihme_recent',
    'Suicide': 'suicide_recent',
    'Alcohol': 'alcohol_recent',
    'Poisoning': 'poison_recent',
    'RoadTraffic': 'traffic_recent',
    'Homicide': 'homicide_recent',
    'MaternalMortality': 'maternal_recent',
    'Childhood': 'u5mr_recent',
    'Diabetes': 'diabetes_ihme_recent',
    'DrugDisorder': 'drug_disorders_recent',
    'UnintentionalInjury': 'unintentional_injuries_ihme_recent',
    'Neoplasms': 'neoplasms_ihme_recent',
}

# Get list of indicators from predictor_dfs
indicator_names = list(predictor_dfs.keys())

# Create tables for each indicator showing top 5 and bottom 5 countries
indicator_extremes = {}

for indicator_name in indicator_names:
    # Get the recent dataframe for this indicator
    if indicator_name in indicator_to_recent_df:
        recent_var_name = indicator_to_recent_df[indicator_name]
        if recent_var_name in globals():
            df_recent = globals()[recent_var_name]
        else:
            continue
    else:
        continue
    
    # Find Male and Female columns dynamically (they end with _Male or _Female)
    male_cols = [col for col in df_recent.columns if col.endswith('_Male')]
    female_cols = [col for col in df_recent.columns if col.endswith('_Female')]
    
    # Get the base name from the first male column (remove _Male suffix)
    if male_cols and female_cols:
        # Use the first matching pair
        male_col = male_cols[0]
        female_col = female_cols[0]
        # Verify they have the same base name
        base_name_male = male_col[:-5]  # Remove '_Male'
        base_name_female = female_col[:-7]  # Remove '_Female'
        if base_name_male == base_name_female:
            # Create summary for this indicator
            indicator_data = df_recent[[male_col, female_col]].copy()
            indicator_data['Country'] = df_recent['Country']
            
            # For male values
            male_sorted = indicator_data.sort_values(male_col, ascending=False)
            male_top5 = male_sorted.head(5)[['Country', male_col]].copy()
            male_bottom5 = male_sorted.tail(5)[['Country', male_col]].copy()
            
            # For female values
            female_sorted = indicator_data.sort_values(female_col, ascending=False)
            female_top5 = female_sorted.head(5)[['Country', female_col]].copy()
            female_bottom5 = female_sorted.tail(5)[['Country', female_col]].copy()
            
            indicator_extremes[indicator_name] = {
                'male_top5': male_top5,
                'male_bottom5': male_bottom5,
                'female_top5': female_top5,
                'female_bottom5': female_bottom5
            }

# Display results for a few key indicators
key_indicators = ['Alcohol', 'Homicide', 'Cardiovascular', 'Suicide']
for indicator in key_indicators:
    if indicator in indicator_extremes:
        print(f"\n{'='*60}")
        print(f"{indicator} - Extreme Values")
        print(f"{'='*60}")
        print(f"\nTop 5 Countries - Male Values:")
        display(indicator_extremes[indicator]['male_top5'])
        print(f"\nBottom 5 Countries - Male Values:")
        display(indicator_extremes[indicator]['male_bottom5'])
        print(f"\nTop 5 Countries - Female Values:")
        display(indicator_extremes[indicator]['female_top5'])
        print(f"\nBottom 5 Countries - Female Values:")
        display(indicator_extremes[indicator]['female_bottom5'])

#### For Gender Gaps: Largest and Smallest Gaps

In [ ]:
gap_extremes = {}

for indicator_name in indicator_names:
    if indicator_name in indicator_to_recent_df:
        recent_var_name = indicator_to_recent_df[indicator_name]
        
        # Get the recent dataframe (already filtered to OECD and most recent year)
        if recent_var_name in globals():
            df_recent = globals()[recent_var_name]
            
            # Find columns dynamically
            gap_cols = [col for col in df_recent.columns if col.startswith('Gap_')]
            male_cols = [col for col in df_recent.columns if col.endswith('_Male')]
            female_cols = [col for col in df_recent.columns if col.endswith('_Female')]
            
            # Find matching columns (same base name)
            if gap_cols and male_cols and female_cols:
                # Get the base name from the gap column (remove 'Gap_' prefix)
                gap_col = gap_cols[0]
                base_name = gap_col[4:]  # Remove 'Gap_' prefix
                
                # Find matching male and female columns
                male_col = f'{base_name}_Male'
                female_col = f'{base_name}_Female'
                
                if male_col in df_recent.columns and female_col in df_recent.columns:
                    # Create summary with country names
                    gap_data = df_recent[[male_col, female_col, gap_col]].copy()
                    gap_data['Country'] = df_recent['Country']
                    
                    # Sort by gap (largest positive gaps first)
                    # Note: Gap is computed as Male - Female, so positive means males have higher rates
                    gap_sorted = gap_data.sort_values(gap_col, ascending=False)
                    
                    gap_top5 = gap_sorted.head(5).copy()
                    gap_bottom5 = gap_sorted.tail(5).copy()
                    
                    gap_extremes[indicator_name] = {
                        'top5': gap_top5,
                        'bottom5': gap_bottom5
                    }

# Display results for key indicators
for indicator in key_indicators:
    if indicator in gap_extremes:
        print(f"\n{'='*60}")
        print(f"{indicator} - Gender Gap Extremes (Male - Female)")
        print(f"{'='*60}")
        gap_info = gap_extremes[indicator]
        # Get column names dynamically from the DataFrame
        top5_df = gap_info['top5']
        # Find Male, Female, and Gap columns
        male_col = [col for col in top5_df.columns if col.endswith('_Male')][0] if any(col.endswith('_Male') for col in top5_df.columns) else None
        female_col = [col for col in top5_df.columns if col.endswith('_Female')][0] if any(col.endswith('_Female') for col in top5_df.columns) else None
        gap_col = [col for col in top5_df.columns if col.startswith('Gap_')][0] if any(col.startswith('Gap_') for col in top5_df.columns) else None
        
        cols_to_show = ['Country']
        if male_col:
            cols_to_show.append(male_col)
        if female_col:
            cols_to_show.append(female_col)
        if gap_col:
            cols_to_show.append(gap_col)
        
        print(f"\nTop 5 Countries - Largest Gaps (Male > Female):")
        display(gap_info['top5'][cols_to_show])
        print(f"\nBottom 5 Countries - Smallest Gaps (Female > Male):")
        display(gap_info['bottom5'][cols_to_show])

#### For HALE Gap: All Countries Ranked

In [ ]:
# Create ranked table of all countries by HALE gap
hale_ranked = analysis_complete[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].copy()
hale_ranked = hale_ranked.sort_values('HALE_gap', ascending=False).reset_index()
hale_ranked['Rank'] = range(1, len(hale_ranked) + 1)
# Rename: Country (code) -> Code, add Country (name)
hale_ranked = hale_ranked.rename(columns={'Country': 'Code'})
hale_ranked['Country'] = hale_ranked['Code'].map(code_to_who_country)

# Create output version without Rank column (Rank kept internally for comparison table)
hale_ranked_output = hale_ranked[['Country', 'HALE_Years_Male', 
                                   'HALE_Years_Female', 'HALE_gap']].copy()

print("All Countries Ranked by HALE Gap (Female - Male)")
print("="*80)
hale_ranked_output

In [ ]:
# Write HALE gap by country table to HTML (without Rank column)
write_html_table(hale_ranked_output, "jb/tables/hale_gap_by_country.html")

In [ ]:
# Summary statistics for HALE gap
print("\nHALE Gap Summary Statistics:")
print("="*50)
print(f"Mean HALE Gap: {hale_ranked['HALE_gap'].mean():.2f} years")
print(f"Median HALE Gap: {hale_ranked['HALE_gap'].median():.2f} years")
print(f"Standard Deviation: {hale_ranked['HALE_gap'].std():.2f} years")
print(f"Range: {hale_ranked['HALE_gap'].min():.2f} to {hale_ranked['HALE_gap'].max():.2f} years")
print(f"\nCountries with Largest HALE Gap (Top 5):")
hale_ranked_output.head(5)
print(f"\nCountries with Smallest HALE Gap (Bottom 5):")
hale_ranked_output.tail(5)

#### For Life Expectancy Gap: All Countries Ranked

In [ ]:
# Create ranked table of all countries by Life Expectancy gap
le_ranked = analysis_complete[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].copy()
le_ranked = le_ranked.sort_values('LifeExpectancy_gap', ascending=False).reset_index()
le_ranked['Rank'] = range(1, len(le_ranked) + 1)
# Rename: Country (code) -> Code, add Country (name)
le_ranked = le_ranked.rename(columns={'Country': 'Code'})
le_ranked['Country'] = le_ranked['Code'].map(code_to_who_country)

# Create output version without Rank column (Rank kept internally for comparison table)
le_ranked_output = le_ranked[['Country', 'LifeExpectancy_Years_Male', 
                              'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].copy()

print("All Countries Ranked by Life Expectancy Gap (Female - Male)")
print("="*80)
le_ranked_output

In [ ]:
# Write Life Expectancy gap by country table to HTML (without Rank column)
write_html_table(le_ranked_output, "jb/tables/le_gap_by_country.html")

In [ ]:
# Summary statistics for Life Expectancy gap
print("\nLife Expectancy Gap Summary Statistics:")
print("="*50)
print(f"Mean Life Expectancy Gap: {le_ranked['LifeExpectancy_gap'].mean():.2f} years")
print(f"Median Life Expectancy Gap: {le_ranked['LifeExpectancy_gap'].median():.2f} years")
print(f"Standard Deviation: {le_ranked['LifeExpectancy_gap'].std():.2f} years")
print(f"Range: {le_ranked['LifeExpectancy_gap'].min():.2f} to {le_ranked['LifeExpectancy_gap'].max():.2f} years")
print(f"\nCountries with Largest Life Expectancy Gap (Top 5):")
le_ranked_output.head(5)
print(f"\nCountries with Smallest Life Expectancy Gap (Bottom 5):")
le_ranked_output.tail(5)

#### Comparison of HALE and Life Expectancy Rankings

In [ ]:
# Compare HALE and Life Expectancy rankings
comparison_ranked = pd.DataFrame({
    'Country': hale_ranked['Country'],
    'HALE_Gap': hale_ranked['HALE_gap'],
    'HALE_Rank': hale_ranked['Rank'],
    'LifeExpectancy_Gap': le_ranked['LifeExpectancy_gap'],
    'LE_Rank': le_ranked['Rank'],
    'Rank_Difference': hale_ranked['Rank'] - le_ranked['Rank']
}).sort_values('HALE_Gap', ascending=False)

# Create output version without rank columns
comparison_ranked_output = comparison_ranked[['Country', 'HALE_Gap', 'LifeExpectancy_Gap', 'Rank_Difference']].copy()

print("="*80)
print("Comparison: HALE Gap vs Life Expectancy Gap Rankings")
print("="*80)
print(f"Correlation between rankings: {comparison_ranked['HALE_Rank'].corr(comparison_ranked['LE_Rank']):.3f}")
comparison_ranked_output

In [ ]:
# Write comparison table to HTML (without rank columns)
write_html_table(comparison_ranked_output, "jb/tables/gap_comparison_by_country.html")

### Step 2.4: Summary Statistics by Indicator

In [ ]:
# Create summary table with rate and gap statistics for each indicator
def compute_indicator_stats(indicator_name, recent_var_name):
    """Compute statistics for a single indicator."""
    if recent_var_name not in globals():
        return None
    
    df_recent = globals()[recent_var_name]
    
    # Get OECD countries only for consistency
    df_oecd = get_oecd(df_recent)
    
    # Find Mid and Gap columns dynamically
    mid_cols = [col for col in df_recent.columns if col.startswith('Mid_')]
    gap_cols = [col for col in df_recent.columns if col.startswith('Gap_')]
    
    # Handle normal case: indicators with both Mid and Gap columns
    if mid_cols and gap_cols:
        mid_col = mid_cols[0]
        gap_col = gap_cols[0]
        
        # Extract midpoint and gap values
        midpoints = df_oecd[mid_col].dropna()
        gaps = df_oecd[gap_col].dropna()
        
        if len(midpoints) > 0 and len(gaps) > 0:
            return {
                'Indicator': indicator_name,
                'Median Rate': midpoints.median(),
                'Min Rate': midpoints.min(),
                'Max Rate': midpoints.max(),
                'Median Gap': gaps.median(),
                'Min Gap': gaps.min(),
                'Max Gap': gaps.max()
            }
    
    # Handle special case: MaternalMortality (female-only, no Gap/Mid columns)
    # Use the Female value as both midpoint and gap
    elif indicator_name == 'MaternalMortality':
        female_cols = [col for col in df_recent.columns if col.endswith('_Female')]
        if female_cols:
            female_col = female_cols[0]
            values = df_oecd[female_col].dropna()
            
            if len(values) > 0:
                return {
                    'Indicator': indicator_name,
                    'Median Rate': values.median(),
                    'Min Rate': values.min(),
                    'Max Rate': values.max(),
                    'Median Gap': -values.median(),
                    'Min Gap': -values.min(),
                    'Max Gap': -values.max()
                }
    
    return None

In [ ]:
# Process predictor indicators
predictor_summary = []
for indicator_name, recent_var_name in indicator_to_recent_df.items():
    stats = compute_indicator_stats(indicator_name, recent_var_name)
    if stats:
        predictor_summary.append(stats)

# Process target variables
target_summary = []
target_indicators = {
    'HALE': 'hale_recent',
    'Life Expectancy': 'le_recent'
}
for indicator_name, recent_var_name in target_indicators.items():
    stats = compute_indicator_stats(indicator_name, recent_var_name)
    if stats:
        target_summary.append(stats)

In [ ]:
# Create DataFrames (keep Indicator as a column, not index)
predictor_df = pd.DataFrame(predictor_summary)
target_df = pd.DataFrame(target_summary)

# Split predictor table into rates and gaps
predictor_rates = predictor_df[['Indicator', 'Median Rate', 'Min Rate', 'Max Rate']].copy()
predictor_gaps = predictor_df[['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']].copy()

# Calculate correlations with target variables
# For rates: use Mid_ columns, for gaps: use Gap_ columns
# Special case: MaternalMortality uses Female column for both
predictor_rates['Corr HALE'] = np.nan
predictor_rates['Corr LE'] = np.nan
predictor_gaps['Corr HALE'] = np.nan
predictor_gaps['Corr LE'] = np.nan

for idx, indicator in enumerate(predictor_rates['Indicator']):
    if indicator == 'MaternalMortality':
        # MaternalMortality: use Female column for both rates and gaps
        female_col = 'MaternalMortality_Female'
        predictor_rates.loc[idx, 'Corr HALE'] = predictors[female_col].corr(target_hale)
        predictor_rates.loc[idx, 'Corr LE'] = predictors[female_col].corr(target_le)
        predictor_gaps.loc[idx, 'Corr HALE'] = predictors[female_col].corr(target_hale)
        predictor_gaps.loc[idx, 'Corr LE'] = predictors[female_col].corr(target_le)
    else:
        # Find the corresponding Mid_ and Gap_ columns in predictors DataFrame
        mid_col = f'Mid_{indicator}'
        gap_col = f'Gap_{indicator}'
        
        # Calculate correlations for rates (using Mid_ columns)
        predictor_rates.loc[idx, 'Corr HALE'] = predictors[mid_col].corr(target_hale)
        predictor_rates.loc[idx, 'Corr LE'] = predictors[mid_col].corr(target_le)
        
        # Calculate correlations for gaps (using Gap_ columns)
        predictor_gaps.loc[idx, 'Corr HALE'] = predictors[gap_col].corr(target_hale)
        predictor_gaps.loc[idx, 'Corr LE'] = predictors[gap_col].corr(target_le)

# Split target table into rates and gaps, reverse gap signs, and adjust column names
target_rates = target_df[['Indicator', 'Median Rate', 'Min Rate', 'Max Rate']].copy()
target_rates.columns = ['Indicator', 'Median', 'Min', 'Max']
target_gaps = target_df[['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']].copy()
# Reverse sign of gaps (from Male - Female to Female - Male)
target_gaps['Median Gap'] = -target_gaps['Median Gap']
target_gaps['Min Gap'] = -target_gaps['Min Gap']
target_gaps['Max Gap'] = -target_gaps['Max Gap']
# After negation, min and max are swapped, so swap the column values
target_gaps['Min Gap'], target_gaps['Max Gap'] = target_gaps['Max Gap'], target_gaps['Min Gap']
target_gaps.columns = ['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']

In [ ]:
print("Predictor Indicators - Rates:")
predictor_rates = predictor_rates.sort_values(by='Median Rate', ascending=False).reset_index(drop=True)
predictor_rates

In [ ]:
print("Predictor Indicators - Gaps:")
predictor_gaps = predictor_gaps.sort_values(by='Median Gap', ascending=False).reset_index(drop=True)
predictor_gaps

In [ ]:
print("Target Variables - Rates:")
target_rates.sort_values(by='Median', ascending=False).reset_index(drop=True)

In [ ]:
print("Target Variables - Gaps:")
target_gaps

In [ ]:
write_html_table(predictor_rates, "jb/tables/predictor_rates.html")
write_html_table(predictor_gaps,  "jb/tables/predictor_gaps.html")

write_html_table(target_rates,    "jb/tables/target_rates.html")
write_html_table(target_gaps,     "jb/tables/target_gaps.html")

In [ ]:
# Create table showing correlation between Rate (Mid) and Gap for each indicator
rate_gap_correlations = []

for indicator in predictor_rates['Indicator']:
    # Skip MaternalMortality - it doesn't have a Gap column (female-only)
    if indicator == 'MaternalMortality':
        continue
    
    # Find the corresponding Mid_ and Gap_ columns in predictors DataFrame
    rate_col = f'Mid_{indicator}'
    gap_col = f'Gap_{indicator}'
    
    # Calculate correlation between rate and gap
    if rate_col in predictors.columns and gap_col in predictors.columns:
        corr = predictors[rate_col].corr(predictors[gap_col])
        rate_gap_correlations.append({
            'Indicator': indicator,
            'Correlation': corr
        })

# Create DataFrame and format (keep Indicator as a column)
rate_gap_corr_df = pd.DataFrame(rate_gap_correlations)

# Sort by correlation (descending)
rate_gap_corr_df = rate_gap_corr_df.sort_values('Correlation', ascending=False).reset_index(drop=True)

In [ ]:
write_html_table(rate_gap_corr_df, "jb/tables/rate_gap_correlation.html")

In [ ]:
# Create table showing top 10 correlations between rates (Mid columns)
# Get all Mid_ columns
mid_cols = [col for col in predictors.columns if col.startswith('Mid_')]

# Calculate correlation matrix for rates
rates_corr_matrix = predictors[mid_cols].corr()

# Extract upper triangle (excluding diagonal) and convert to list of pairs
rates_corr_pairs = []
for i in range(len(rates_corr_matrix.columns)):
    for j in range(i+1, len(rates_corr_matrix.columns)):
        indicator1 = rates_corr_matrix.columns[i].replace('Mid_', '')
        indicator2 = rates_corr_matrix.columns[j].replace('Mid_', '')
        corr_val = rates_corr_matrix.iloc[i, j]
        rates_corr_pairs.append({
            'Rate 1': indicator1,
            'Rate 2': indicator2,
            'Correlation': corr_val
        })

# Create DataFrame and sort by absolute correlation
rates_corr_df = pd.DataFrame(rates_corr_pairs)
rates_corr_df['Abs Correlation'] = rates_corr_df['Correlation'].abs()
rates_corr_df = rates_corr_df.sort_values('Abs Correlation', ascending=False).head(10)

# Drop the absolute value column
rates_corr_df = rates_corr_df[['Rate 1', 'Rate 2', 'Correlation']].reset_index(drop=True)

In [ ]:
write_html_table(rates_corr_df,     "jb/tables/rate_rate_correlation_top10.html")

In [ ]:
# Create table showing top 10 correlations between gaps (Gap columns)
# Get all Gap_ columns (excluding MaternalMortality which doesn't have a Gap column)
gap_cols = [col for col in predictors.columns if col.startswith('Gap_')]

# Calculate correlation matrix for gaps
gaps_corr_matrix = predictors[gap_cols].corr()

# Extract upper triangle (excluding diagonal) and convert to list of pairs
gaps_corr_pairs = []
for i in range(len(gaps_corr_matrix.columns)):
    for j in range(i+1, len(gaps_corr_matrix.columns)):
        indicator1 = gaps_corr_matrix.columns[i].replace('Gap_', '')
        indicator2 = gaps_corr_matrix.columns[j].replace('Gap_', '')
        corr_val = gaps_corr_matrix.iloc[i, j]
        gaps_corr_pairs.append({
            'Gap 1': indicator1,
            'Gap 2': indicator2,
            'Correlation': corr_val
        })

# Create DataFrame and sort by absolute correlation
gaps_corr_df = pd.DataFrame(gaps_corr_pairs)
gaps_corr_df['Abs Correlation'] = gaps_corr_df['Correlation'].abs()
gaps_corr_df = gaps_corr_df.sort_values('Abs Correlation', ascending=False).head(10)

# Drop the absolute value column
gaps_corr_df = gaps_corr_df[['Gap 1', 'Gap 2', 'Correlation']].reset_index(drop=True)

In [ ]:
write_html_table(gaps_corr_df,     "jb/tables/gap_gap_correlation_top10.html")

### Step 2.5: Save Data for Future Use

In [ ]:
# Save predictors and target to HDF5 file for future use
import pandas as pd

# Save to HDF5 file
hdf_file = '../data/hale_analysis_data.h5'
with pd.HDFStore(hdf_file, mode='w') as store:
    store['predictors'] = predictors
    store['target_hale'] = target_hale
    store['target_le'] = target_le

In [ ]:
predictors.columns

In [ ]:
from utils import beep

beep()